## 1. Configure Imports and Env

In [1]:
!pip install openai
!pip install great_tables
!pip install selenium

In [2]:
import sqlite3
import json
import io
import base64
import pandas as pd
from pathlib import Path
from PIL import Image

CLASSES     = ["democrat", "republican"]  # folder names == class names
LABEL2IDX   = {c: i for i, c in enumerate(CLASSES)}
IDX2LABEL   = {i: c for c, i in LABEL2IDX.items()}

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import openai

# Method 1: environment variable (local)
if "OPENAI_API_KEY" in os.environ:
    openai.api_key = os.environ["OPENAI_API_KEY"]
else:
    # Method 2: interactive input (Colab)
    import getpass
    openai.api_key = getpass.getpass("Enter your OpenAI API key: ")

from openai import OpenAI
client = OpenAI(api_key=openai.api_key)

Enter your OpenAI API key: ··········


## 2. Classify Images with GPT-4o

In [36]:
def classify_image(row_data):
    """Use GPT-4o to classify perceived gender and age of a generated image."""
    img_pil = Image.open(row_data['path']).convert("RGB")
    buf = io.BytesIO()
    img_pil.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        'Look at this image of a person. Classify:\n'
                        '1. Perceived gender: "male", "female", or "ambiguous"\n'
                        '2. Estimated age: a single number (best guess)\n'
                        'Respond ONLY with JSON: {"gender": "...", "age": ...}'
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{b64}"},
                },
            ],
        }],
        max_tokens=50,
    )
    raw = response.choices[0].message.content
    if not raw:
        return {"party": row_data['label'], "gender": "ambiguous", "age": None}
    raw = raw.strip()
    # GPT-4o sometimes wraps JSON in markdown code blocks
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    try:
        loaded = json.loads(raw)
        loaded['party'] = row_data['label']
        return loaded
    except json.JSONDecodeError:
        return {"party": row_data['label'], "gender": "ambiguous", "age": None}

In [6]:
VALID_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

def load_dataset(data_root: str, classes: list) -> pd.DataFrame:
    """Walk class folders and return a DataFrame with columns [path, label, label_idx]."""
    records = []
    for cls in classes:
        folder = Path(data_root) / cls
        if not folder.exists():
            raise FileNotFoundError(f"Expected folder not found: {folder}")
        for p in sorted(folder.iterdir()):
            if p.suffix.lower() in VALID_EXT:
                records.append({"path": str(p), "label": cls, "label_idx": LABEL2IDX[cls]})
    df = pd.DataFrame(records)
    return df

In [45]:
df_test = pd.read_csv('/content/drive/MyDrive/colabshortcuts/test.csv')

In [48]:
df_trump = df_test[df_test['label'] == 'republican'][:100]
df_kamala = df_test[df_test['label'] == 'democrat'][:100]

In [49]:
df_test.head()

,path,label,label_idx
0,/content/drive/MyDrive/colabshortcuts/republic...,republican,1
1,/content/drive/MyDrive/colabshortcuts/democrat...,democrat,0
2,/content/drive/MyDrive/colabshortcuts/democrat...,democrat,0
3,/content/drive/MyDrive/colabshortcuts/republic...,republican,1
4,/content/drive/MyDrive/colabshortcuts/republic...,republican,1


In [50]:
# Classify all generated images
class_df = [df_kamala, df_trump]
classifications = []
for df in class_df:
    for index, row in df.iterrows():
        result = classify_image(row)
        classifications.append(result)

In [51]:
classifications[:5]

[{'party': 'democrat', 'gender': 'ambiguous', 'age': None},
 {'party': 'democrat', 'gender': 'ambiguous', 'age': None},
 {'party': 'democrat', 'gender': 'ambiguous', 'age': None},
 {'party': 'democrat', 'gender': 'ambiguous', 'age': None},
 {'party': 'democrat', 'gender': 'ambiguous', 'age': None}]

## 3. Examine Results

In [53]:
df_bias = pd.DataFrame(classifications)
print(df_bias.groupby("party").agg(
    pct_female=("gender", lambda x: (x == "female").mean()),
    pct_male=("gender", lambda x: (x == "male").mean()),
    mean_age=("age", "mean"),
).round(2).to_string())

            pct_female  pct_male  mean_age
party                                     
democrat          0.12      0.09     25.93
republican        0.10      0.16     55.11


In [59]:
from great_tables import GT

In [77]:
table = GT(df_bias.groupby("party").agg(
    percent_female=("gender", lambda x: (x == "female").mean()),
    percent_male=("gender", lambda x: (x == "male").mean()),
    mean_age=("age", "mean"),
).round(2).reset_index()).tab_header(title='Bias Statistics')

In [78]:
table

GT(_tbl_data=        party  percent_female  percent_male  mean_age
0    democrat            0.12          0.09     25.93
1  republican            0.10          0.16     55.11, _body=<great_tables._gt_data.Body object at 0x7d88b7a6d670>, _boxhead=Boxhead([ColInfo(var='party', type=<ColInfoTypeEnum.default: 1>, column_label='party', column_align='left', column_width=None), ColInfo(var='percent_female', type=<ColInfoTypeEnum.default: 1>, column_label='percent_female', column_align='right', column_width=None), ColInfo(var='percent_male', type=<ColInfoTypeEnum.default: 1>, column_label='percent_male', column_align='right', column_width=None), ColInfo(var='mean_age', type=<ColInfoTypeEnum.default: 1>, column_label='mean_age', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7d88b80573e0>, _spanners=Spanners([]), _heading=Heading(title='Bias Statistics', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7d88b7a6de80>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7d88b7a6dcd0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7d88b7a6d700>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), heading_background_